# Artificial Intelligence Approach to Predict Student Behaviour and Performance

This notebook trains **two models** on a single dataset:

1. **Student Behaviour Prediction (Classification)** → predicts `behavior_label` (**At-Risk / Neutral / Engaged**)
2. **Student Performance Prediction (Regression + Classification)**  
   - Regression predicts `final_score` (0–100)  
   - Optional classification predicts `performance_label` (**Low / Medium / High**)

✅ It also saves the trained models + preprocessing artifacts to a `models/` folder.


In [1]:

# If you're running locally, install dependencies (uncomment):
# !pip install -U pandas numpy scikit-learn joblib matplotlib

import os
from pathlib import Path
import numpy as np
import pandas as pd

DATA_PATH = Path(r"D:\avanthi\student behaviour and performance\student_behavior_performance_dataset.csv")
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

print("Dataset:", DATA_PATH)
print("Models dir:", MODELS_DIR.resolve())


Dataset: D:\avanthi\student behaviour and performance\student_behavior_performance_dataset.csv
Models dir: D:\avanthi\student behaviour and performance\models


In [2]:

# --------------------------
# 1) Load dataset
# --------------------------
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())

target_behavior = "behavior_label"
target_score = "final_score"
target_perf_class = "performance_label"

# Features: drop targets + student_id
drop_cols = ["student_id", target_behavior, target_score, target_perf_class]
X = df.drop(columns=drop_cols)
y_beh = df[target_behavior]
y_score = df[target_score]
y_perf = df[target_perf_class]

print("Features:", X.shape[1])
print(X.dtypes.value_counts())


Shape: (2500, 23)


,student_id,gender,age,parent_education,family_income_band,travel_time_mins,study_hours_per_week,attendance_rate,past_failures,tutoring,...,extracurricular_hours,discipline_incidents,late_submissions,participation_score,quiz_avg,assignment_avg,midterm_score,behavior_label,final_score,performance_label
0,1,F,16,none,middle,44,6.7,0.889,0,1,...,3.19,0,1,32.4,52.4,62.9,47.3,Engaged,60.3,Medium
1,2,M,15,bachelor,low,10,9.1,0.854,0,0,...,3.23,1,1,78.3,56.0,43.6,100.0,Neutral,53.0,Medium
2,3,F,20,bachelor,middle,15,10.0,0.896,2,0,...,4.35,0,1,77.5,52.9,53.9,29.5,At-Risk,43.8,Low
3,4,F,20,bachelor,middle,55,11.3,0.819,0,0,...,4.31,1,3,65.6,48.9,85.0,30.9,At-Risk,47.8,Low
4,5,M,21,bachelor,high,23,16.7,0.801,1,0,...,1.86,0,0,53.2,70.3,54.5,28.0,Engaged,47.4,Low


Features: 19
float64    9
int64      7
object     3
Name: count, dtype: int64


In [4]:

# Quick sanity checks
print("Behavior label distribution:")
display(y_beh.value_counts(normalize=True).rename("share").to_frame())

print("Performance class distribution:")
display(y_perf.value_counts(normalize=True).rename("share").to_frame())

print("Final score summary:")
display(y_score.describe())


Behavior label distribution:


,share
behavior_label,
Neutral,0.34
Engaged,0.33
At-Risk,0.33


Performance class distribution:


,share
performance_label,
Medium,0.7168
High,0.1492
Low,0.1340


Final score summary:


count    2500.000000
mean       63.099840
std        11.380595
min        23.400000
25%        55.500000
50%        63.100000
75%        70.625000
max        96.000000
Name: final_score, dtype: float64

In [5]:

# --------------------------
# 2) Train/test split
# --------------------------
from sklearn.model_selection import train_test_split

X_train, X_test, yb_train, yb_test = train_test_split(
    X, y_beh, test_size=0.2, random_state=42, stratify=y_beh
)

Xs_train, Xs_test, ys_train, ys_test = train_test_split(
    X, y_score, test_size=0.2, random_state=42
)

Xp_train, Xp_test, yp_train, yp_test = train_test_split(
    X, y_perf, test_size=0.2, random_state=42, stratify=y_perf
)

print("Behaviour split:", X_train.shape, X_test.shape)
print("Score split:", Xs_train.shape, Xs_test.shape)
print("Perf-class split:", Xp_train.shape, Xp_test.shape)


Behaviour split: (2000, 19) (500, 19)
Score split: (2000, 19) (500, 19)
Perf-class split: (2000, 19) (500, 19)


In [6]:

# --------------------------
# 3) Preprocessing pipeline
#    - OneHot for categorical
#    - StandardScaler for numeric
# --------------------------
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop"
)

print("Categorical:", cat_cols)
print("Numeric:", num_cols)


Categorical: ['gender', 'parent_education', 'family_income_band']
Numeric: ['age', 'travel_time_mins', 'study_hours_per_week', 'attendance_rate', 'past_failures', 'tutoring', 'internet_access', 'sleep_hours', 'screen_time_hours', 'extracurricular_hours', 'discipline_incidents', 'late_submissions', 'participation_score', 'quiz_avg', 'assignment_avg', 'midterm_score']


In [8]:

# --------------------------
# 4) Behaviour Model (Classification)
# --------------------------
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

beh_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", RandomForestClassifier(
        n_estimators=350,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

beh_model.fit(X_train, yb_train)
pred_beh = beh_model.predict(X_test)

print(classification_report(yb_test, pred_beh, digits=3))
print("Confusion matrix:", confusion_matrix(yb_test, pred_beh))


              precision    recall  f1-score   support

     At-Risk      0.641     0.648     0.645       165
     Engaged      0.624     0.685     0.653       165
     Neutral      0.421     0.376     0.398       170

    accuracy                          0.568       500
   macro avg      0.562     0.570     0.565       500
weighted avg      0.561     0.568     0.563       500

Confusion matrix: [[107  11  47]
 [ 11 113  41]
 [ 49  57  64]]


In [10]:

# --------------------------
# 5) Performance Model (Regression: predict final_score)
# --------------------------
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

score_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("reg", RandomForestRegressor(
        n_estimators=450,
        random_state=42,
        n_jobs=-1
    ))
])

score_model.fit(Xs_train, ys_train)
pred_score = score_model.predict(Xs_test)

mae = mean_absolute_error(ys_test, pred_score)
mse = mean_squared_error(ys_test, pred_score)
rmse = mse ** 0.5
r2 = r2_score(ys_test, pred_score)

print(f"MAE:  {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R^2:  {r2:.3f}")


MAE:  5.387
RMSE: 6.817
R^2:  0.661


In [11]:

# Optional: Performance classification model (Low/Medium/High)
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

perf_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", GradientBoostingClassifier(random_state=42))
])

perf_model.fit(Xp_train, yp_train)
pred_perf = perf_model.predict(Xp_test)

print("Accuracy:", accuracy_score(yp_test, pred_perf))
print(classification_report(yp_test, pred_perf, digits=3))


Accuracy: 0.796
              precision    recall  f1-score   support

        High      0.721     0.413     0.525        75
         Low      0.733     0.493     0.589        67
      Medium      0.811     0.933     0.868       358

    accuracy                          0.796       500
   macro avg      0.755     0.613     0.661       500
weighted avg      0.787     0.796     0.779       500



In [12]:

# --------------------------
# 6) Save models (joblib)
# --------------------------
import joblib

BEH_PATH = MODELS_DIR / "student_behavior_classifier.joblib"
SCORE_PATH = MODELS_DIR / "student_performance_regressor.joblib"
PERF_PATH = MODELS_DIR / "student_performance_classifier.joblib"

joblib.dump(beh_model, BEH_PATH)
joblib.dump(score_model, SCORE_PATH)
joblib.dump(perf_model, PERF_PATH)

print("Saved:")
print(" -", BEH_PATH.resolve())
print(" -", SCORE_PATH.resolve())
print(" -", PERF_PATH.resolve())


Saved:
 - D:\avanthi\student behaviour and performance\models\student_behavior_classifier.joblib
 - D:\avanthi\student behaviour and performance\models\student_performance_regressor.joblib
 - D:\avanthi\student behaviour and performance\models\student_performance_classifier.joblib


In [13]:

# --------------------------
# 7) Inference function (use both models)
# --------------------------
def predict_student(sample: dict):
    """sample: dict with the SAME feature names as X columns (no targets)."""
    x_df = pd.DataFrame([sample])
    beh = beh_model.predict(x_df)[0]
    beh_proba = None
    if hasattr(beh_model.named_steps["clf"], "predict_proba"):
        beh_proba = beh_model.predict_proba(x_df)[0]
        beh_classes = list(beh_model.named_steps["clf"].classes_)
        beh_proba = dict(zip(beh_classes, beh_proba))

    score = float(score_model.predict(x_df)[0])
    perf = perf_model.predict(x_df)[0]

    return {
        "behavior_label": beh,
        "behavior_probabilities": beh_proba,
        "final_score_pred": round(score, 2),
        "performance_label": perf
    }

# Example: pick a random row from test set
example = X_test.sample(1, random_state=7).iloc[0].to_dict()
result = predict_student(example)
result


{'behavior_label': 'Neutral',
 'behavior_probabilities': {'At-Risk': 0.3514285714285714,
  'Engaged': 0.2714285714285714,
  'Neutral': 0.37714285714285717},
 'final_score_pred': 60.31,
 'performance_label': 'Medium'}

In [14]:
import joblib
beh_model = joblib.load("models/student_behavior_classifier.joblib")


## What you get in this project

### Dataset file
- `student_behavior_performance_dataset.csv`

### Saved model files
- `models/student_behavior_classifier.joblib`
- `models/student_performance_regressor.joblib`
- `models/student_performance_classifier.joblib`

You can load a saved model later like this:
```python
import joblib
beh_model = joblib.load("models/student_behavior_classifier.joblib")
```
